# Day 10: Outliers & Comparative Analysis

## Section 1: Outlier Detection (Z-Score)

In [ ]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv("../data/sales_cleaned.csv")

# 2. Calculate Z-Score for Sales
# Z = (Value - Mean) / Standard Deviation
df['z_score'] = (df['Sales'] - df['Sales'].mean()) / df['Sales'].std()

# 3. Identify Outliers (Usually anything > 3 or < -3)
outliers = df[np.abs(df['z_score']) > 3]

print(f"Detected {len(outliers)} outliers in the dataset.")
print(outliers[['Date', 'Product', 'Sales', 'z_score']])

## Section 2: The Comparison Matrix (Pivot Tables)

In [ ]:
# Create a matrix comparing Sales across Regions and Categories
pivot_comparison = df.pivot_table(
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc='sum'
)

# Calculate the percentage difference between two columns if applicable
pivot_comparison['Growth'] = (pivot_comparison['Electronics'] - pivot_comparison['Accessories']) / pivot_comparison['Accessories']

print("Regional Category Performance:")
print(pivot_comparison)

## Section 3: Clustered Bar Chart

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.barplot(x='Region', y='Sales', hue='Category', data=df, estimator=sum)
plt.title("Revenue Comparison: Region vs. Category")
plt.show()

## Section 4: The "Cleaned" Forecast

### Dirty Forecast (with outliers)

In [ ]:
import datetime as dt
from sklearn.linear_model import LinearRegression

# Load fresh data with z-scores already computed
df = pd.read_csv("../data/sales_cleaned.csv")
df['Date'] = pd.to_datetime(df['Date'])
df['z_score'] = (df['Sales'] - df['Sales'].mean()) / df['Sales'].std()

# --- DIRTY FORECAST (includes outliers) ---
df['Date_Ordinal'] = df['Date'].map(dt.datetime.toordinal)
X_dirty = df[['Date_Ordinal']]
y_dirty = df['Sales']

model_dirty = LinearRegression()
model_dirty.fit(X_dirty, y_dirty)
print(f"Dirty Trend Coefficient: {model_dirty.coef_[0]:.4f}")

last_date = df['Date_Ordinal'].max()
future_dates = np.array(range(last_date + 1, last_date + 31)).reshape(-1, 1)
future_preds_dirty = model_dirty.predict(future_dates)

plt.figure(figsize=(10, 6))
plt.scatter(df['Date'], df['Sales'], color='blue', label='Actual Sales')
future_dates_dt = [dt.datetime.fromordinal(int(d)) for d in future_dates.flatten()]
plt.plot(future_dates_dt, future_preds_dirty, color='red', linestyle='--', label='Dirty Forecast')
plt.title("Dirty Forecast (With Outliers)")
plt.legend()
plt.show()

### Cleaned Forecast (outliers removed)

In [ ]:
# --- CLEANED FORECAST (outliers removed) ---
df_clean = df[np.abs(df['z_score']) <= 3]
print(f"Removed {len(df) - len(df_clean)} outliers from the dataset.")

X_clean = df_clean[['Date_Ordinal']]
y_clean = df_clean['Sales']

model_clean = LinearRegression()
model_clean.fit(X_clean, y_clean)
print(f"Clean Trend Coefficient: {model_clean.coef_[0]:.4f}")

future_preds_clean = model_clean.predict(future_dates)

plt.figure(figsize=(10, 6))
plt.scatter(df_clean['Date'], df_clean['Sales'], color='green', label='Cleaned Sales')
plt.plot(future_dates_dt, future_preds_clean, color='orange', linestyle='--', label='Cleaned Forecast')
plt.title("Cleaned Forecast (Outliers Removed)")
plt.legend()
plt.show()

### Side-by-Side Comparison

In [ ]:
# --- SIDE-BY-SIDE COMPARISON ---
plt.figure(figsize=(12, 6))
plt.scatter(df['Date'], df['Sales'], color='blue', alpha=0.5, label='All Sales (incl. outliers)')
plt.scatter(df_clean['Date'], df_clean['Sales'], color='green', alpha=0.7, label='Cleaned Sales')
plt.plot(future_dates_dt, future_preds_dirty, color='red', linestyle='--', linewidth=2, label=f'Dirty Forecast (coef={model_dirty.coef_[0]:.2f})')
plt.plot(future_dates_dt, future_preds_clean, color='orange', linestyle='--', linewidth=2, label=f'Clean Forecast (coef={model_clean.coef_[0]:.2f})')
plt.title("Impact of Outliers: Dirty vs. Cleaned Forecast")
plt.legend()
plt.show()

print(f"\nDirty Trend Coefficient: {model_dirty.coef_[0]:.4f}")
print(f"Clean Trend Coefficient: {model_clean.coef_[0]:.4f}")
print(f"Difference: {abs(model_dirty.coef_[0] - model_clean.coef_[0]):.4f}")

## Reflection

**When is an outlier a 'mistake' to be deleted, and when is it a 'discovery' that should be investigated?**

An outlier is a **mistake** when it results from data entry errors, system glitches, or processing bugs. For example, a sales record showing $18,750 for a Mouse product that normally sells for $50-$400 is almost certainly a data error — perhaps a decimal point was misplaced or multiple transactions were merged into one row.

An outlier is a **discovery** when it reflects a genuine, explainable real-world event. For example, if MeetMux suddenly sees a 10x spike in signups on a single day, that could look like an outlier — but investigating it might reveal that a viral social media post or a major press mention drove real organic traffic. Deleting that data point would erase evidence of a breakthrough moment.

**The rule of thumb:** Never delete an outlier without investigating its cause first. Always keep an Anomalies Report so that "signals" are not lost. In our dataset, the two outliers (Laptop at $15,230 and Mouse at $18,750) have z-scores above 3, strongly suggesting data errors since they are far outside the normal range for those products. But if we were tracking MeetMux signups and saw a sudden spike, we would investigate before removing it — that spike could be the start of a new growth trend.